# ⚡ Week 5: Production NLP Sentiment Analysis Project
**Dataset:** Amazon Product Reviews Dataset (`DataSet (W5).csv`)  
**Objective:** Build an end-to-end sentiment classification pipeline using TF-IDF feature extraction and benchmarked machine learning models (Logistic Regression, Naive Bayes, Linear SVM).


In [1]:
import os
import re
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

import sys
sys.path.append('..')
from utils.preprocess import load_and_detect_dataset, clean_text_pipeline
from utils.visualization import plot_sentiment_distribution, plot_review_length_distribution, generate_wordclouds, plot_confusion_matrix, plot_model_comparison

print('All libraries imported successfully!')

## 1. Data Loading & Dynamic Inspection
Using our dataset auto-loader to automatically locate the CSV file and infer text and target columns.

In [2]:
df, text_col, target_col, meta = load_and_detect_dataset('../data/DataSet (W5).csv')
print(f'Dataset Shape: {df.shape}')
print(f'Detected Review Text Column: {text_col}')
print(f'Detected Target Sentiment Column: {target_col}')
print('\nData Head:')
display(df.head(5))
print('\nMissing Values Count:\n', df.isnull().sum())
print('\nClass Distribution:\n', df[target_col].value_counts())

## 2. Text Preprocessing & Cleaning Pipeline
Applying lowercasing, HTML/URL removal, punctuation stripping, stopword filtering with negation preservation, and NLTK lemmatization.

In [3]:
start_time = time.time()
df['clean_text'] = df[text_col].apply(clean_text_pipeline)
df = df[df['clean_text'].str.strip() != ''].copy()
print(f'Cleaned {len(df)} non-empty reviews in {time.time() - start_time:.2f} seconds.')
print('\nSample Cleaned Text Output:')
for i in range(3):
    print(f'RAW: {df[text_col].iloc[i]}')
    print(f'CLEANED: {df["clean_text"].iloc[i]}\n')

## 3. Exploratory Data Analysis (EDA)
Visualizing class distribution, review length metrics, and Positive/Negative Word Clouds.

In [4]:
fig_dist = plot_sentiment_distribution(df, target_col)
plt.show()

fig_len = plot_review_length_distribution(df, text_col, target_col)
plt.show()

fig_pos, fig_neg = generate_wordclouds(df, 'clean_text', target_col)
plt.show()

## 4. Feature Extraction - TF-IDF Vectorization
Transforming text into numeric term frequency-inverse document frequency vectors with sublinear scaling.

In [5]:
vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)
X = vectorizer.fit_transform(df['clean_text'])
y = df[target_col].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
print(f'TF-IDF Feature Matrix Shape: {X.shape}')
print(f'Train shape: {X_train.shape}, Test shape: {X_test.shape}')

## 5 & 6. Model Training, Benchmarking & Evaluation
Training Logistic Regression, Multinomial Naive Bayes, and Calibrated Linear SVM models.

In [6]:
models = {
    'Logistic Regression': LogisticRegression(C=1.0, solver='liblinear', random_state=42),
    'Multinomial Naive Bayes': MultinomialNB(alpha=0.1),
    'Linear SVM': CalibratedClassifierCV(LinearSVC(C=1.0, random_state=42, max_iter=2000))
}

results = []
for name, model in models.items():
    t0 = time.time()
    model.fit(X_train, y_train)
    train_t = time.time() - t0
    
    p0 = time.time()
    y_pred = model.predict(X_test)
    pred_t = time.time() - p0
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    
    results.append({
        'Model': name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1,
        'Train Time (s)': round(train_t, 4),
        'Pred Time (s)': round(pred_t, 4)
    })
    
    print(f'=== {name} ===')
    print(classification_report(y_test, y_pred))
    cm = confusion_matrix(y_test, y_pred)
    plot_confusion_matrix(cm, model_name=name)
    plt.show()

res_df = pd.DataFrame(results)
display(res_df)
plot_model_comparison(res_df)
plt.show()

## 7. Model Selection & Persistence
Selecting the top F1-Score model and persisting artifacts using `joblib`.

In [7]:
best_row = res_df.sort_values(by='F1-Score', ascending=False).iloc[0]
best_name = best_row['Model']
print(f'🏆 Best Model Selected: {best_name} (F1: {best_row["F1-Score"]:.4f})')

best_model = models[best_name]
os.makedirs('../models', exist_ok=True)
joblib.dump(best_model, '../models/best_model.pkl')
joblib.dump(vectorizer, '../models/tfidf_vectorizer.pkl')
print('Successfully saved model and vectorizer to ../models/ directory!')